# 01 — HTTP source → MinIO (RAW bucket)

**RF-01**: download the official NYC yellow taxi Parquet over HTTP and land it
in a MinIO bucket using **dlt**.

Memory note: the source holds ~3.5M rows. It is streamed to disk once and then
handed to dlt in **PyArrow record batches**, which keeps memory flat.

All dependencies come from `requirements.txt` — nothing is installed here.

In [1]:
from __future__ import annotations

import sys
import tempfile
from pathlib import Path

sys.path.insert(0, "/home/jovyan/scripts")

import dlt
import pyarrow as pa
import pyarrow.parquet as pq
import requests

import config as cfg

PIPELINE_NAME = "http_to_minio"


In [2]:
def download(url: str, destination: Path) -> Path:
    """Stream the source file to local disk."""
    print(f"Downloading {url}", flush=True)
    with requests.get(url, stream=True, timeout=120) as response:
        response.raise_for_status()
        total = int(response.headers.get("Content-Length", 0))
        written = 0
        with destination.open("wb") as handle:
            for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                handle.write(chunk)
                written += len(chunk)
                if total:
                    print(f"  {written / total:6.1%}  ({written / 1e6:,.0f} MB)", flush=True)
    print(f"Downloaded {destination.stat().st_size / 1e6:,.1f} MB", flush=True)
    return destination


In [3]:
@dlt.resource(name=cfg.RAW_TABLE, write_disposition="replace")
def yellow_tripdata():
    """Yield the source Parquet as PyArrow batches.

    `write_disposition="replace"` makes this idempotent: re-running the
    notebook does not duplicate rows in the RAW bucket.
    """
    with tempfile.TemporaryDirectory() as tmp:
        local = download(cfg.SOURCE_URL, Path(tmp) / "source.parquet")

        parquet_file = pq.ParquetFile(local)
        total_rows = parquet_file.metadata.num_rows
        print(f"Source rows: {total_rows:,}", flush=True)

        emitted = 0
        for batch in parquet_file.iter_batches(batch_size=cfg.BATCH_ROWS):
            emitted += batch.num_rows
            print(f"  -> {emitted:,} / {total_rows:,} rows", flush=True)
            yield pa.Table.from_batches([batch])


In [4]:
cfg.banner("01 | HTTP -> MinIO (RAW bucket)")
print(f"Source     : {cfg.SOURCE_URL}")
print(f"Destination: s3://{cfg.RAW_BUCKET}/{cfg.RAW_DATASET}/{cfg.RAW_TABLE}/")
print(f"MinIO      : {cfg.MINIO_ENDPOINT}")

pipeline = dlt.pipeline(
    pipeline_name=PIPELINE_NAME,
    destination="filesystem",
    dataset_name=cfg.RAW_DATASET,
)

load_info = pipeline.run(yellow_tripdata(), loader_file_format="parquet")
print(load_info)



  01 | HTTP -> MinIO (RAW bucket)
Source     : https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-01.parquet
Destination: s3://nyc-taxi-raw/raw/yellow_tripdata/
MinIO      : http://minio:9000


2026-09-03 02:17:37,020|[INFO]|4594|136263733958080|dlt|pipeline.py|_restore_state_from_destination:1947|The state was restored from the destination filesystem (dlt.destinations.filesystem):raw


   14.2%  (8 MB)
   28.4%  (17 MB)
   42.5%  (25 MB)
   56.7%  (34 MB)
   70.9%  (42 MB)
   85.1%  (50 MB)
   99.3%  (59 MB)
  100.0%  (59 MB)
Downloaded 59.2 MB
Source rows: 3,475,226
  -> 250,000 / 3,475,226 rows
  -> 500,000 / 3,475,226 rows
  -> 750,000 / 3,475,226 rows
  -> 1,000,000 / 3,475,226 rows
  -> 1,250,000 / 3,475,226 rows
  -> 1,500,000 / 3,475,226 rows
  -> 1,750,000 / 3,475,226 rows
  -> 2,000,000 / 3,475,226 rows
  -> 2,250,000 / 3,475,226 rows
  -> 2,500,000 / 3,475,226 rows
  -> 2,750,000 / 3,475,226 rows
  -> 3,000,000 / 3,475,226 rows
  -> 3,250,000 / 3,475,226 rows
  -> 3,475,226 / 3,475,226 rows


2026-09-03 02:17:42,613|[INFO]|4594|136263733958080|dlt|pool_runner.py|create_pool:203|Created none pool with 1 workers
2026-09-03 02:17:42,614|[INFO]|4594|136263733958080|dlt|normalize.py|run:309|Running file normalizing
2026-09-03 02:17:42,615|[INFO]|4594|136263733958080|dlt|normalize.py|run:312|Found 1 load packages
2026-09-03 02:17:42,617|[INFO]|4594|136263733958080|dlt|normalize.py|run:335|Found 1 files in schema http_to_minio load_id 1788401857.0521398
2026-09-03 02:17:42,621|[INFO]|4594|136263733958080|dlt|normalize.py|spool_schema_files:298|Created new load package 1788401857.0521398 on loading volume with 1 files
2026-09-03 02:17:42,626|[INFO]|4594|136263733958080|dlt|worker.py|_get_items_normalizer:137|A file format for table yellow_tripdata was specified to parquet in the resource so parquet format being used.
2026-09-03 02:17:42,626|[INFO]|4594|136263733958080|dlt|worker.py|_get_items_normalizer:186|Created items normalizer ArrowItemsNormalizer with writer ArrowToParquetWri

Pipeline http_to_minio load step finished in 0.42 seconds
1 load package(s) were loaded to destination filesystem and into dataset raw
The filesystem destination used s3://nyc-taxi-raw location to store data
Load package 1788401857.0521398 is LOADED and contains no failed jobs


...

2026-09-03 01:17:15,052|[INFO]|218|134604633261504|dlt|load.py|start_new_jobs:327|Will load additional 0, creating jobs


2026-09-03 01:17:15,061|[INFO]|218|134604633261504|dlt|load.py|complete_package:657|All jobs completed, archiving package 1788398228.3012493 with aborted set to False


2026-09-03 01:17:15,062|[INFO]|218|134604633261504|dlt|pool_runner.py|run_pool:279|Closing processing pool


2026-09-03 01:17:15,063|[INFO]|218|134604633261504|dlt|pool_runner.py|run_pool:282|Processing pool closed


Pipeline http_to_minio load step finished in 0.51 seconds
1 load package(s) were loaded to destination filesystem and into dataset raw
The filesystem destination used s3://nyc-taxi-raw location to store data
Load package 1788398228.3012493 is LOADED and contains no failed jobs


In [5]:
cfg.banner("Verification | objects in the RAW bucket")
filesystem = cfg.minio_filesystem()
prefix = f"{cfg.RAW_BUCKET}/{cfg.RAW_DATASET}/{cfg.RAW_TABLE}"
objects = cfg.list_parquet(filesystem, prefix)

assert objects, f"FAIL: no Parquet file found under s3://{prefix}/"

rows = 0
for key in objects:
    size = filesystem.info(key)["size"]
    with filesystem.open(key, "rb") as handle:
        rows += pq.ParquetFile(handle).metadata.num_rows
    print(f"  {key}  ({size / 1e6:,.1f} MB)")

print(f"\nRows landed in MinIO: {rows:,}")
assert rows == cfg.EXPECTED_ROWS, f"expected {cfg.EXPECTED_ROWS:,} rows, found {rows:,}"
print("OK - RAW bucket matches the expected row count.")



  Verification | objects in the RAW bucket
  nyc-taxi-raw/raw/yellow_tripdata/1788401857.0521398.08e971fb2c.parquet  (72.9 MB)

Rows landed in MinIO: 3,475,226
OK - RAW bucket matches the expected row count.


  nyc-taxi-raw/raw/yellow_tripdata/1788398228.3012493.ff5e43ca4f.parquet  (72.9 MB)

Rows landed in MinIO: 3,475,226
OK - RAW bucket matches the expected row count.
